# 📄 Modern Document Processing & Storage
### Turning Unstructured Data into LLM-Ready Inputs

---

Real-world data doesn't come clean. It lives in PDFs, Word documents, PowerPoint decks, scanned images, and spreadsheets. Before an LLM can use any of it, you need a pipeline that:

1. **Extracts** text from raw files (OCR, layout parsing)
2. **Understands** and structures the content (LLM-based extraction)
3. **Chunks** the text into retrievable units
4. **Embeds and stores** it in a vector database

By the end of this notebook, you'll have built that entire pipeline from scratch.

**What you'll learn:**
1. 🔍 OCR & Layout Parsing — `markitdown` and `pymupdf4llm`
2. 🧠 LLM-Based Information Extraction
3. ✂️ Document Chunking Strategies
4. 🗄️ Vector Stores — Amazon S3 Vectors
5. 🏭 End-to-End Document Pipeline

---

## 📦 Install Dependencies

In [ ]:
# Core document processing libraries
!pip install markitdown[all] --quiet       # Microsoft's universal document → Markdown converter
!pip install pymupdf4llm --quiet           # PDF-first extraction with layout analysis
!pip install openai numpy --quiet          # OpenAI API + numerics
!pip install boto3 --quiet                 # AWS SDK (for S3 Vectors)
!pip install python-docx openpyxl --quiet  # For creating demo documents

print("✅ All dependencies installed!")

In [ ]:
import os
import json
import re
import uuid
import textwrap
import numpy as np
from pathlib import Path
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# ── Helpers ───────────────────────────────────────────────────────────────────
def ask(prompt, system="You are a helpful assistant.", model="gpt-4o-mini", temperature=0):
    resp = client.chat.completions.create(
        model=model, temperature=temperature,
        messages=[{"role": "system", "content": system},
                  {"role": "user",   "content": prompt}]
    )
    return resp.choices[0].message.content

def get_embedding(text, model="text-embedding-3-small"):
    resp = client.embeddings.create(input=text, model=model)
    return resp.data[0].embedding

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print("✅ Setup complete!")

---

## 🔍 Section 1: OCR & Layout Parsing

### The Problem with Raw Documents

LLMs consume text — but most business data lives in binary formats:

```
annual_report.pdf     ← scanned image, no selectable text
product_spec.docx     ← formatted Word doc with tables
sales_data.xlsx       ← multi-sheet spreadsheet
pitch_deck.pptx       ← slides with bullets, charts, speaker notes
invoice_scan.png      ← photographed document
```

We need tools that understand **document structure** — not just dump raw text.

### Why Markdown is the Target Format

Markdown is the ideal intermediate format for LLMs because:
- It preserves **structure** (headings, tables, lists)
- It is **token-efficient** (minimal extra characters)
- LLMs are trained on massive amounts of it and understand it natively

---

### Tool 1: `markitdown` — Universal Document Converter

Microsoft's open-source library that converts virtually any file format into clean Markdown in just a few lines of code.

In [ ]:
# First, let's create some sample documents to work with
import json
from pathlib import Path
from docx import Document
import openpyxl

Path("sample_docs").mkdir(exist_ok=True)

# ── Create a sample Word document ─────────────────────────────────────────────
doc = Document()
doc.add_heading("Q1 2025 Sales Report", 0)
doc.add_heading("Executive Summary", 1)
doc.add_paragraph(
    "Total revenue for Q1 2025 reached PHP 12.4 million, "
    "representing a 23% year-over-year growth. The Southeast Asia "
    "region led performance with a 35% increase in new customer acquisition."
)
doc.add_heading("Regional Breakdown", 1)
table = doc.add_table(rows=4, cols=3)
table.style = "Table Grid"
headers = ["Region", "Revenue (PHP M)", "YoY Growth"]
rows_data = [
    ["Philippines", "5.2", "+28%"],
    ["Indonesia",   "4.1", "+19%"],
    ["Vietnam",     "3.1", "+22%"],
]
for i, h in enumerate(headers):
    table.rows[0].cells[i].text = h
for r, row_data in enumerate(rows_data, start=1):
    for c, val in enumerate(row_data):
        table.rows[r].cells[c].text = val
doc.add_heading("Key Highlights", 1)
for point in [
    "Launched 3 new product SKUs in February",
    "Customer retention rate improved to 87%",
    "Operating costs reduced by 8% through automation",
]:
    doc.add_paragraph(point, style="List Bullet")
doc.save("sample_docs/q1_sales_report.docx")

# ── Create a sample Excel spreadsheet ─────────────────────────────────────────
wb = openpyxl.Workbook()
ws = wb.active
ws.title = "Employee Directory"
ws.append(["Name",    "Department",    "Role",             "Salary (PHP)", "Start Date"])
ws.append(["Ana Reyes",    "Engineering",  "Senior Engineer",  85000, "2021-03-15"])
ws.append(["Juan Santos",  "Marketing",    "Brand Manager",    72000, "2022-07-01"])
ws.append(["Maria Cruz",   "Engineering",  "Lead Engineer",   90000, "2020-11-22"])
ws.append(["Paolo Delos",  "Sales",        "Account Executive",65000, "2023-01-10"])
ws.append(["Lea Gomez",    "HR",           "HR Manager",        70000, "2019-06-05"])
wb.save("sample_docs/employees.xlsx")

# ── Create a plain text file ───────────────────────────────────────────────────
Path("sample_docs/meeting_notes.txt").write_text("""
Meeting Notes - Product Roadmap Review
Date: April 15, 2025
Attendees: CEO (Rosa), CTO (Marco), Product Lead (Lia)

DECISIONS:
- Approved budget for AI feature development: PHP 2.5M
- Target launch date for v2.0: September 2025
- Lia will lead the new personalization engine project

ACTION ITEMS:
1. Marco: Prepare technical spec by April 30
2. Rosa: Align with board on revised timeline
3. Lia: Hire 2 additional ML engineers by May 15

RISKS IDENTIFIED:
- Dependency on third-party data provider (medium risk)
- Potential delay if ML engineers not onboarded on time (high risk)
""")

print("✅ Sample documents created in sample_docs/")
print([f.name for f in Path("sample_docs").iterdir()])

In [ ]:
from markitdown import MarkItDown

# ── Initialize MarkItDown ──────────────────────────────────────────────────────
# With LLM client: enables image description and OCR for image-based PDFs
md_converter = MarkItDown(
    llm_client=client,
    llm_model="gpt-4o-mini"
)

# ── Convert Word document ──────────────────────────────────────────────────────
result = md_converter.convert("sample_docs/q1_sales_report.docx")

print("=" * 60)
print("WORD DOC → MARKDOWN")
print("=" * 60)
print(result.text_content)

In [ ]:
# ── Convert Excel spreadsheet ──────────────────────────────────────────────────
result_xl = md_converter.convert("sample_docs/employees.xlsx")

print("=" * 60)
print("EXCEL → MARKDOWN")
print("=" * 60)
print(result_xl.text_content)

In [ ]:
# ── Convert plain text ─────────────────────────────────────────────────────────
result_txt = md_converter.convert("sample_docs/meeting_notes.txt")

print("=" * 60)
print("TXT → MARKDOWN")
print("=" * 60)
print(result_txt.text_content)

### Supported Formats by `markitdown`

| Format | Notes |
|---|---|
| PDF | Text-based; use OCR plugin for scanned pages |
| DOCX | Full support: headings, tables, lists |
| XLSX / XLS | Sheets become Markdown tables |
| PPTX | Slides, speaker notes, embedded images |
| HTML | Cleaned, structured extraction |
| URL | Fetches and converts live web pages |
| Images (PNG, JPG) | Requires `llm_client` for OCR description |
| Audio (MP3, WAV) | Speech transcription via LLM |
| ZIP | Extracts and converts all contents |

---

### Tool 2: `pymupdf4llm` — PDF-First with Layout Intelligence

While `markitdown` is a great universal tool, `pymupdf4llm` is purpose-built for **complex PDF documents**. It understands:
- Multi-column layouts and correct reading order
- Tables embedded in text flow
- Headers and footers to exclude
- Selective OCR (only for pages that need it — ~50% faster)

In [ ]:
import pymupdf4llm
import pymupdf

# ── Create a sample PDF using pymupdf ─────────────────────────────────────────
doc = pymupdf.open()  # new empty document
page = doc.new_page()

# Build a simple PDF with mixed content
page.insert_text((50, 50),  "TECHNICAL SPECIFICATION — AI PLATFORM v2.0", fontsize=16)
page.insert_text((50, 90),  "Section 1: Overview", fontsize=13)
page.insert_text((50, 120), "This document outlines the architecture and requirements for", fontsize=11)
page.insert_text((50, 140), "the AI Platform v2.0, targeting production deployment in Q3 2025.", fontsize=11)
page.insert_text((50, 180), "Section 2: System Requirements", fontsize=13)
page.insert_text((50, 210), "• Python 3.10+", fontsize=11)
page.insert_text((50, 230), "• 16 GB RAM minimum (32 GB recommended)", fontsize=11)
page.insert_text((50, 250), "• CUDA-compatible GPU (optional for inference)", fontsize=11)
page.insert_text((50, 270), "• AWS account with S3 and Bedrock access", fontsize=11)
page.insert_text((50, 310), "Section 3: API Endpoints", fontsize=13)

# Draw a simple table
table_data = [
    ["Endpoint",        "Method", "Description"],
    ["/v1/embed",       "POST",   "Generate embeddings"],
    ["/v1/search",      "POST",   "Semantic search"],
    ["/v1/extract",     "POST",   "Extract structured data"],
    ["/v1/summarize",   "POST",   "Summarize a document"],
]
x_start, y_start = 50, 340
col_widths = [160, 60, 220]
row_height = 22

for r, row in enumerate(table_data):
    x = x_start
    for c, cell in enumerate(row):
        rect = pymupdf.Rect(x, y_start + r * row_height,
                            x + col_widths[c], y_start + (r + 1) * row_height)
        page.draw_rect(rect, color=(0, 0, 0), width=0.5)
        page.insert_text((x + 4, y_start + r * row_height + 15), cell, fontsize=9)
        x += col_widths[c]

page.insert_text((50, 490), "Section 4: Data Retention Policy", fontsize=13)
page.insert_text((50, 515), "All embeddings stored in S3 Vectors are retained for 90 days", fontsize=11)
page.insert_text((50, 535), "by default. Administrators can configure custom retention windows", fontsize=11)
page.insert_text((50, 555), "via the management console.", fontsize=11)

doc.save("sample_docs/tech_spec.pdf")
doc.close()
print("✅ Sample PDF created: sample_docs/tech_spec.pdf")

In [ ]:
# ── Basic PDF → Markdown extraction ───────────────────────────────────────────
md_text = pymupdf4llm.to_markdown("sample_docs/tech_spec.pdf")

print("=" * 60)
print("PDF → MARKDOWN (pymupdf4llm)")
print("=" * 60)
print(md_text)

In [ ]:
# ── Page chunking — each page becomes a dict with metadata ───────────────────
page_chunks = pymupdf4llm.to_markdown(
    "sample_docs/tech_spec.pdf",
    page_chunks=True,         # Return list of dicts (one per page)
    header=False,             # Exclude repeated page headers
    footer=False,             # Exclude repeated page footers
)

print(f"Pages extracted: {len(page_chunks)}")
for i, chunk in enumerate(page_chunks):
    print(f"\n--- Page {i+1} ---")
    print(f"Metadata: {chunk.get('metadata', {})}")
    print(f"Text preview: {chunk['text'][:300]}...")

### When to Use Which Tool?

| Scenario | Best Tool |
|---|---|
| Mixed document types (PDF, DOCX, XLSX, PPTX) | `markitdown` |
| Complex PDFs (multi-column, scanned pages, tables) | `pymupdf4llm` |
| Web pages or URLs | `markitdown` |
| PDF page-by-page chunking with metadata | `pymupdf4llm` |
| Audio/image files | `markitdown` (with LLM client) |
| LangChain / LlamaIndex integration | `pymupdf4llm` (native loaders) |

---

## 🧠 Section 2: LLM-Based Information Extraction

Once you have clean Markdown text, you can use an LLM to extract **structured information** from it — turning unstructured prose into typed, queryable data.

### The Pattern

```
Raw Document
    ↓ (markitdown / pymupdf4llm)
Markdown Text
    ↓ (LLM + structured prompt)
JSON / Structured Data
    ↓ (your application)
Database / API / Dashboard
```

In [ ]:
# ── Step 1: Extract the Markdown from the Word doc ────────────────────────────
report_md = md_converter.convert("sample_docs/q1_sales_report.docx").text_content
notes_md  = md_converter.convert("sample_docs/meeting_notes.txt").text_content

print("✅ Markdown extracted from documents")

In [ ]:
# ── Example 1: Extract KPIs from a financial report ───────────────────────────

kpi_prompt = f"""
You are a financial data extraction assistant.
Extract all key performance indicators (KPIs) from the document below.

Return ONLY a valid JSON object — no markdown backticks, no explanation.
Use this exact structure:
{{
  "document_title": "string",
  "period": "string",
  "total_revenue": "string",
  "yoy_growth_pct": "number",
  "regions": [
    {{"name": "string", "revenue": "string", "growth": "string"}}
  ],
  "highlights": ["string"]
}}

--- DOCUMENT ---
{report_md}
"""

raw = ask(kpi_prompt)
kpis = json.loads(raw)

print("✅ Extracted KPIs:")
print(json.dumps(kpis, indent=2))

In [ ]:
# ── Example 2: Extract action items from meeting notes ────────────────────────

actions_prompt = f"""
Extract all action items from these meeting notes.
Return ONLY valid JSON — no extra text.

{{
  "meeting_date": "string",
  "attendees": ["string"],
  "decisions": ["string"],
  "action_items": [
    {{"owner": "string", "task": "string", "deadline": "string or null"}}
  ],
  "risks": [
    {{"description": "string", "severity": "low|medium|high"}}
  ]
}}

--- MEETING NOTES ---
{notes_md}
"""

raw2 = ask(actions_prompt)
actions = json.loads(raw2)

print("✅ Extracted Meeting Data:")
print(json.dumps(actions, indent=2))

In [ ]:
# ── Example 3: Multi-document extraction with schema enforcement ──────────────
# Useful when processing batches of documents

def extract_structured(markdown_text: str, schema: dict, doc_type: str) -> dict:
    """
    Extract structured data from a markdown document using a JSON schema.
    """
    prompt = f"""
You are a document data extraction specialist.
Extract information from the {doc_type} below.
Return ONLY valid JSON matching exactly this schema:

{json.dumps(schema, indent=2)}

Use null for missing fields. Do not add extra keys.

--- DOCUMENT ---
{markdown_text[:3000]}  
"""
    raw = ask(prompt)
    try:
        # Strip markdown fences if the model adds them
        clean = re.sub(r"```json|```", "", raw).strip()
        return json.loads(clean)
    except json.JSONDecodeError:
        return {"error": "Failed to parse", "raw": raw}


# Define schema for a tech spec document
tech_spec_schema = {
    "title": "string",
    "version": "string",
    "target_date": "string or null",
    "requirements": ["string"],
    "api_endpoints": [{"path": "string", "method": "string", "description": "string"}],
    "data_retention_days": "number or null"
}

tech_spec_md = pymupdf4llm.to_markdown("sample_docs/tech_spec.pdf")
result = extract_structured(tech_spec_md, tech_spec_schema, "technical specification")

print("✅ Extracted from PDF:")
print(json.dumps(result, indent=2))

---

## ✂️ Section 3: Document Chunking Strategies

### Why Chunk?

You can't feed an entire document to a vector search system as one unit — it would:
- Dilute the meaning (embedding tries to represent everything at once)
- Miss specific details in retrieval
- Exceed context window limits

Chunking breaks your documents into **retrievable units** that are small enough to be semantically precise but large enough to contain useful context.

### Chunking Strategies Compared

| Strategy | How | Best For |
|---|---|---|
| **Fixed-size** | Split every N characters/tokens | Simple, predictable pipelines |
| **Sentence** | Split at sentence boundaries | General prose documents |
| **Paragraph** | Split at paragraph breaks | Articles, reports |
| **Semantic / Header-based** | Split at Markdown headings | Structured documents, tech specs |
| **Page-based** | One chunk per page | PDFs with distinct pages |
| **Sliding window** | Overlap between chunks | Preserves cross-boundary context |

In [ ]:
# ── Strategy 1: Fixed-Size Chunking with Overlap ───────────────────────────────

def fixed_size_chunk(text: str, chunk_size: int = 500, overlap: int = 50) -> list[dict]:
    """
    Split text into fixed-size character chunks with optional overlap.
    Overlap helps preserve context across chunk boundaries.
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk_text = text[start:end]
        chunks.append({
            "id": f"chunk_{len(chunks)+1}",
            "text": chunk_text,
            "strategy": "fixed_size",
            "char_start": start,
            "char_end": end,
        })
        start += chunk_size - overlap  # step back by overlap amount
    return chunks

sample_text = md_converter.convert("sample_docs/q1_sales_report.docx").text_content
fixed_chunks = fixed_size_chunk(sample_text, chunk_size=300, overlap=50)

print(f"Fixed-size chunking → {len(fixed_chunks)} chunks")
for c in fixed_chunks[:3]:
    print(f"\n  [{c['id']}] chars {c['char_start']}–{c['char_end']}")
    print(f"  {repr(c['text'][:80])}...")

In [ ]:
# ── Strategy 2: Sentence-Based Chunking ──────────────────────────────────────

def sentence_chunk(text: str, sentences_per_chunk: int = 3, overlap: int = 1) -> list[dict]:
    """
    Split text by sentences, grouping N sentences per chunk.
    """
    # Simple sentence splitter — use nltk or spacy for production
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s.strip() for s in sentences if s.strip()]
    
    chunks = []
    i = 0
    while i < len(sentences):
        group = sentences[i : i + sentences_per_chunk]
        chunks.append({
            "id": f"chunk_{len(chunks)+1}",
            "text": " ".join(group),
            "strategy": "sentence",
            "sentence_start": i,
            "sentence_end": i + len(group) - 1,
        })
        i += sentences_per_chunk - overlap
    return chunks

sentence_chunks = sentence_chunk(sample_text, sentences_per_chunk=3, overlap=1)
print(f"Sentence chunking → {len(sentence_chunks)} chunks")
for c in sentence_chunks[:3]:
    print(f"\n  [{c['id']}] sentences {c['sentence_start']}–{c['sentence_end']}")
    print(f"  {c['text'][:120]}...")

In [ ]:
# ── Strategy 3: Semantic / Header-Based Chunking (Best for Markdown) ──────────

def header_chunk(markdown_text: str, max_chunk_chars: int = 1000) -> list[dict]:
    """
    Split Markdown at heading boundaries (# ## ###).
    Respects document structure for the highest semantic coherence.
    Falls back to fixed-size if a section is too long.
    """
    # Split at markdown headings
    sections = re.split(r'(?=\n#{1,3} )', markdown_text)
    sections = [s.strip() for s in sections if s.strip()]
    
    chunks = []
    for section in sections:
        # Extract heading for metadata
        heading_match = re.match(r'^(#{1,3})\s+(.*?)\n', section)
        heading = heading_match.group(2) if heading_match else "Introduction"
        level   = len(heading_match.group(1)) if heading_match else 0
        
        if len(section) <= max_chunk_chars:
            chunks.append({
                "id": f"chunk_{len(chunks)+1}",
                "text": section,
                "strategy": "header",
                "heading": heading,
                "heading_level": level,
            })
        else:
            # Section too long — split further by fixed size
            sub = fixed_size_chunk(section, chunk_size=max_chunk_chars, overlap=100)
            for s in sub:
                s["heading"] = heading
                s["heading_level"] = level
                s["id"] = f"chunk_{len(chunks)+1}"
                chunks.append(s)
    return chunks


# Use on the tech spec (a well-structured Markdown document)
spec_md = pymupdf4llm.to_markdown("sample_docs/tech_spec.pdf")
header_chunks = header_chunk(spec_md)

print(f"Header-based chunking → {len(header_chunks)} chunks")
for c in header_chunks:
    print(f"\n  [{c['id']}] H{c['heading_level']}: '{c['heading']}'")
    print(f"  {len(c['text'])} chars — preview: {c['text'][:100]}...")

In [ ]:
# ── Strategy 4: pymupdf4llm Page-Based Chunking ───────────────────────────────
# Each page becomes a chunk — great for PDFs with distinct content per page

page_based_chunks = pymupdf4llm.to_markdown(
    "sample_docs/tech_spec.pdf",
    page_chunks=True,
)

# Normalize into our standard chunk format
normalized_page_chunks = [
    {
        "id": f"page_{i+1}",
        "text": page["text"],
        "strategy": "page",
        "page_number": i + 1,
        "metadata": page.get("metadata", {}),
    }
    for i, page in enumerate(page_based_chunks)
]

print(f"Page-based chunking → {len(normalized_page_chunks)} chunks")
for c in normalized_page_chunks:
    print(f"  Page {c['page_number']}: {len(c['text'])} chars")

In [ ]:
# ── Chunking Strategy Comparison ─────────────────────────────────────────────
# Which strategy produces the most useful chunks for your document?

strategies = {
    "Fixed-size (300 chars, 50 overlap)": fixed_size_chunk(sample_text, 300, 50),
    "Sentence (3 per chunk, 1 overlap)": sentence_chunk(sample_text, 3, 1),
    "Header-based (max 1000 chars)": header_chunk(spec_md, 1000),
    "Page-based": normalized_page_chunks,
}

print("=" * 55)
print(f"{'Strategy':<38} {'# Chunks':>8} {'Avg Chars':>9}")
print("=" * 55)
for name, chunks in strategies.items():
    avg = int(np.mean([len(c["text"]) for c in chunks]))
    print(f"{name:<38} {len(chunks):>8} {avg:>9}")

print("\n💡 Tip: More chunks = finer retrieval granularity, but higher storage + latency.")
print("   Fewer, larger chunks = broader context per result.")

---

## 🗄️ Section 4: Vector Stores — Amazon S3 Vectors

### What is a Vector Store?

A vector store is a database that stores embeddings and can **efficiently find similar vectors** using Approximate Nearest Neighbor (ANN) search — much faster than comparing every vector individually.

```
Query: "What is the data retention policy?"
    ↓ embed
[0.12, -0.43, 0.87, ...]  ← 1536-dimensional vector
    ↓ ANN search
Top-K most similar stored vectors → source text chunks
```

### Amazon S3 Vectors (2025)

AWS launched **S3 Vectors** — the first cloud object store with native vector support. It integrates directly with your existing S3 infrastructure:

- **Sub-second** similarity search across millions of vectors
- **Up to 90% cheaper** than traditional purpose-built vector DBs
- Supports **metadata filtering** alongside vector search
- Native boto3 `s3vectors` client
- **90%+ average recall** for most datasets

### Key Concepts

| Concept | Description | Analogy |
|---|---|---|
| **Vector Bucket** | Top-level container | Like an S3 bucket, but for vectors |
| **Vector Index** | A searchable index within a bucket | Like a database table |
| **Vector** | An embedding + metadata + key | Like a row in a table |
| **Query** | Find top-K similar vectors | Like a semantic search |

> **Note:** S3 Vectors is available in preview. You'll need an AWS account, IAM permissions for S3 Vectors, and `boto3` installed.

In [ ]:
# ── In-Memory Vector Store (for local development / testing) ──────────────────
# This simulates the S3 Vectors interface locally — useful when you don't have
# AWS credentials set up. Section 4b shows the real S3 Vectors implementation.

class LocalVectorStore:
    """
    A minimal in-memory vector store that mirrors the S3 Vectors API pattern:
    put_vectors(), query_vectors(), list_vectors(), delete_vectors()
    
    Use this for local development. Replace with boto3 s3vectors for production.
    """
    def __init__(self, index_name: str, dimension: int):
        self.index_name = index_name
        self.dimension  = dimension
        self._store: dict[str, dict] = {}  # key → {vector, metadata}
        print(f"✅ LocalVectorStore initialized: '{index_name}' ({dimension}d)")

    def put_vectors(self, vectors: list[dict]):
        """Insert or update vectors. Each dict: {key, data: {float32: [...]}, metadata: {...}}"""
        for v in vectors:
            self._store[v["key"]] = {
                "vector": np.array(v["data"]["float32"]),
                "metadata": v.get("metadata", {})
            }
        print(f"  ✅ Inserted {len(vectors)} vectors. Total: {len(self._store)}")

    def query_vectors(self, query_vector: list[float], top_k: int = 5,
                      filter: dict = None, return_metadata: bool = True) -> list[dict]:
        """Find top-K most similar vectors using cosine similarity."""
        q = np.array(query_vector)
        results = []
        for key, item in self._store.items():
            # Optional metadata filter
            if filter:
                if not all(item["metadata"].get(k) == v for k, v in filter.items()):
                    continue
            sim = cosine_similarity(q, item["vector"])
            results.append({"key": key, "distance": sim, "metadata": item["metadata"]})
        results.sort(key=lambda x: x["distance"], reverse=True)
        return results[:top_k]

    def list_vectors(self) -> list[str]:
        return list(self._store.keys())

    def delete_vectors(self, keys: list[str]):
        for k in keys:
            self._store.pop(k, None)
        print(f"  Deleted {len(keys)} vectors.")

# Instantiate
vector_store = LocalVectorStore(
    index_name="docs-index",
    dimension=1536  # text-embedding-3-small outputs 1536 dims
)

In [ ]:
# ── Embed and insert document chunks ──────────────────────────────────────────

# Use the header-based chunks from the tech spec
chunks_to_index = header_chunk(spec_md)

print(f"Embedding {len(chunks_to_index)} chunks...")
vectors_to_insert = []

for i, chunk in enumerate(chunks_to_index):
    embedding = get_embedding(chunk["text"])
    vectors_to_insert.append({
        "key": f"techspec_chunk_{i+1}",
        "data": {"float32": embedding},
        "metadata": {
            "source": "tech_spec.pdf",
            "heading": chunk.get("heading", ""),
            "heading_level": chunk.get("heading_level", 0),
            "text": chunk["text"],  # Store the source text for retrieval
            "doc_type": "technical_specification",
            "chunk_index": i,
        }
    })
    print(f"  [{i+1}/{len(chunks_to_index)}] Embedded: '{chunk.get('heading', 'intro')}'")

vector_store.put_vectors(vectors_to_insert)
print(f"\n✅ {len(vectors_to_insert)} vectors stored!")

In [ ]:
# ── Semantic search against the vector store ──────────────────────────────────

def semantic_search(query: str, top_k: int = 3, filter: dict = None):
    """Search the vector store and display results."""
    print(f"🔍 Query: '{query}'")
    if filter:
        print(f"   Filter: {filter}")
    
    query_embedding = get_embedding(query)
    results = vector_store.query_vectors(
        query_vector=query_embedding,
        top_k=top_k,
        filter=filter,
    )
    
    for rank, result in enumerate(results, 1):
        meta = result["metadata"]
        score = result["distance"]
        print(f"\n  #{rank} [score: {score:.4f}] '{meta.get('heading', 'N/A')}'")
        print(f"  Source: {meta.get('source')}")
        print(f"  Text preview: {meta.get('text', '')[:200]}...")
    return results


# Test queries
results1 = semantic_search("What are the hardware and software requirements?")
print("\n" + "─" * 60 + "\n")
results2 = semantic_search("How long is data kept in storage?")
print("\n" + "─" * 60 + "\n")
results3 = semantic_search("What API endpoints are available?")

In [ ]:
# ── S3 Vectors: Real AWS Implementation ──────────────────────────────────────
# Uncomment and fill in your AWS credentials to use the real S3 Vectors service

"""
import boto3
from botocore.exceptions import ClientError

# Prerequisites:
# 1. AWS account with S3 Vectors access
# 2. Set env vars: AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_DEFAULT_REGION
# 3. pip install boto3

REGION         = "us-east-1"           # S3 Vectors supported regions
BUCKET_NAME    = "my-docs-vector-bucket"  # Must be globally unique
INDEX_NAME     = "tech-specs-index"
DIMENSION      = 1536                  # Must match your embedding model output

# Create clients
s3vectors = boto3.client("s3vectors", region_name=REGION)

# ── 1. Create the vector bucket ──────────────────────────────────────────────
try:
    s3vectors.create_vector_bucket(vectorBucketName=BUCKET_NAME)
    print(f"✅ Vector bucket created: {BUCKET_NAME}")
except ClientError as e:
    if "BucketAlreadyExists" in str(e):
        print(f"Bucket already exists: {BUCKET_NAME}")
    else:
        raise

# ── 2. Create the vector index ───────────────────────────────────────────────
try:
    s3vectors.create_index(
        vectorBucketName=BUCKET_NAME,
        indexName=INDEX_NAME,
        dimension=DIMENSION,
        dataType="float32",
        metadataConfiguration={
            "nonFilterableMetadataKeys": ["text"],  # text is too large to filter on
        }
    )
    print(f"✅ Vector index created: {INDEX_NAME}")
except ClientError as e:
    if "AlreadyExistsException" in str(e):
        print(f"Index already exists: {INDEX_NAME}")
    else:
        raise

# ── 3. Insert vectors in batches ─────────────────────────────────────────────
BATCH_SIZE = 500  # S3 Vectors supports up to 500 vectors per batch

for i in range(0, len(vectors_to_insert), BATCH_SIZE):
    batch = vectors_to_insert[i:i + BATCH_SIZE]
    s3vectors.put_vectors(
        vectorBucketName=BUCKET_NAME,
        indexName=INDEX_NAME,
        vectors=batch
    )
    print(f"  Inserted batch {i // BATCH_SIZE + 1} ({len(batch)} vectors)")

# ── 4. Query the vector index ────────────────────────────────────────────────
query_text = "What are the system requirements?"
query_embedding = get_embedding(query_text)

response = s3vectors.query_vectors(
    vectorBucketName=BUCKET_NAME,
    indexName=INDEX_NAME,
    queryVector={"float32": query_embedding},
    topK=3,
    returnDistance=True,
    returnMetadata=True
)

for result in response["vectors"]:
    print(f"  Score: {result['distance']:.4f}")
    print(f"  Text:  {result['metadata']['text'][:200]}")
    print()

# ── 5. Filtered query (metadata filter) ─────────────────────────────────────
filtered_response = s3vectors.query_vectors(
    vectorBucketName=BUCKET_NAME,
    indexName=INDEX_NAME,
    queryVector={"float32": query_embedding},
    topK=3,
    filter={"doc_type": "technical_specification"},  # Only search this doc type
    returnDistance=True,
    returnMetadata=True
)
"""

print("💡 S3 Vectors code block ready — fill in your AWS credentials to activate.")

### Vector Store Options at a Glance

| Store | Best For | Notes |
|---|---|---|
| **Amazon S3 Vectors** | AWS-native, cost-effective, large scale | Up to 90% cheaper than alternatives |
| **Chroma** | Local dev, prototyping | In-process or persistent mode |
| **Pinecone** | Managed SaaS, production | Generous free tier |
| **pgvector** | Teams already using PostgreSQL | SQL + vectors in one DB |
| **Qdrant** | Self-hosted, advanced filtering | Open source, Docker-friendly |
| **Weaviate** | Multi-modal, hybrid search | Built-in GraphQL API |

> **For RAG (Session 4):** Any store works. The key is consistent embedding models between indexing and querying.

---

## 🏭 Section 5: End-to-End Document Pipeline

Now let's combine everything into a **production-ready pipeline** that processes multiple documents and makes them searchable.

```
┌─────────────────────────────────────────────────────────────┐
│                   DOCUMENT INGESTION PIPELINE               │
│                                                             │
│  Input Files                                                │
│  (PDF, DOCX, XLSX, etc.)                                    │
│        ↓                                                    │
│  ① Parse & Convert  ←── markitdown / pymupdf4llm           │
│        ↓                                                    │
│  ② Extract Structure ←── LLM extraction (optional)         │
│        ↓                                                    │
│  ③ Chunk  ←── header / sentence / fixed-size               │
│        ↓                                                    │
│  ④ Embed  ←── text-embedding-3-small                       │
│        ↓                                                    │
│  ⑤ Store  ←── S3 Vectors / ChromaDB / Pinecone             │
│        ↓                                                    │
│  ⑥ Search  ←── query → top-K chunks → LLM → answer        │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ── The Complete Pipeline as a Reusable Class ─────────────────────────────────

class DocumentPipeline:
    """
    End-to-end pipeline: raw files → searchable vector store.
    
    Stages:
      1. Parse   — convert any file to Markdown
      2. Chunk   — split by headers (or fixed-size fallback)
      3. Embed   — generate embeddings for each chunk
      4. Index   — store in vector store with metadata
    """

    def __init__(self, store: LocalVectorStore, openai_client: OpenAI,
                 chunk_strategy: str = "header"):
        self.store     = store
        self.client    = openai_client
        self.converter = MarkItDown(llm_client=openai_client, llm_model="gpt-4o-mini")
        self.strategy  = chunk_strategy
        self.processed_docs = []

    # ── Stage 1: Parse ────────────────────────────────────────────────────────
    def parse(self, file_path: str) -> str:
        path = Path(file_path)
        if path.suffix.lower() == ".pdf":
            return pymupdf4llm.to_markdown(str(path))
        else:
            return self.converter.convert(str(path)).text_content

    # ── Stage 2: Chunk ────────────────────────────────────────────────────────
    def chunk(self, markdown: str) -> list[dict]:
        if self.strategy == "header":
            return header_chunk(markdown)
        elif self.strategy == "sentence":
            return sentence_chunk(markdown)
        else:
            return fixed_size_chunk(markdown)

    # ── Stage 3+4: Embed and Index ────────────────────────────────────────────
    def ingest(self, file_path: str, tags: dict = None) -> dict:
        """Full ingestion pipeline for a single document."""
        path = Path(file_path)
        print(f"\n📄 Processing: {path.name}")

        # 1. Parse
        print("  → Parsing...")
        markdown = self.parse(file_path)
        print(f"     {len(markdown):,} characters extracted")

        # 2. Chunk
        print(f"  → Chunking ({self.strategy})...")
        chunks = self.chunk(markdown)
        # Filter out empty or very short chunks
        chunks = [c for c in chunks if len(c["text"].strip()) > 50]
        print(f"     {len(chunks)} chunks created")

        # 3. Embed + 4. Index
        print(f"  → Embedding and indexing...")
        vectors = []
        for i, chunk in enumerate(chunks):
            emb = get_embedding(chunk["text"])
            metadata = {
                "source": path.name,
                "file_type": path.suffix.lower().lstrip("."),
                "chunk_index": i,
                "strategy": self.strategy,
                "text": chunk["text"],
                "heading": chunk.get("heading", ""),
                **(tags or {})
            }
            vectors.append({
                "key": f"{path.stem}_{i}_{uuid.uuid4().hex[:6]}",
                "data": {"float32": emb},
                "metadata": metadata,
            })

        self.store.put_vectors(vectors)

        summary = {
            "file": path.name,
            "chars": len(markdown),
            "chunks": len(chunks),
            "vectors_stored": len(vectors),
        }
        self.processed_docs.append(summary)
        print(f"  ✅ Done! {len(vectors)} vectors indexed.")
        return summary

    def ingest_folder(self, folder: str, tags: dict = None) -> list[dict]:
        """Process all supported files in a folder."""
        supported = {".pdf", ".docx", ".xlsx", ".pptx", ".txt", ".md", ".html"}
        results = []
        for f in sorted(Path(folder).iterdir()):
            if f.suffix.lower() in supported:
                results.append(self.ingest(str(f), tags=tags))
        return results

    def search(self, query: str, top_k: int = 3, filter: dict = None) -> list[dict]:
        """Semantic search over all indexed documents."""
        q_emb = get_embedding(query)
        return self.store.query_vectors(q_emb, top_k=top_k, filter=filter)

    def ask_document(self, question: str, top_k: int = 3, filter: dict = None) -> str:
        """Answer a question using retrieved document chunks (preview of RAG)."""
        results = self.search(question, top_k=top_k, filter=filter)
        if not results:
            return "No relevant documents found."

        context = "\n\n---\n\n".join([
            f"Source: {r['metadata']['source']} | Section: {r['metadata'].get('heading', 'N/A')}\n"
            f"{r['metadata']['text']}"
            for r in results
        ])

        prompt = f"""
Answer the question using ONLY the provided document excerpts.
If the answer is not in the excerpts, say "I don't have that information."
Cite the source document in your answer.

Question: {question}

--- DOCUMENT EXCERPTS ---
{context}
"""
        return ask(prompt, temperature=0)

print("✅ DocumentPipeline class defined!")

In [ ]:
# ── Run the full pipeline on all sample documents ─────────────────────────────

# Fresh vector store
pipeline_store = LocalVectorStore(index_name="full-pipeline", dimension=1536)

# Initialize pipeline with header-based chunking
pipeline = DocumentPipeline(
    store=pipeline_store,
    openai_client=client,
    chunk_strategy="header"
)

# Ingest all sample documents
results = pipeline.ingest_folder(
    "sample_docs",
    tags={"project": "demo", "department": "operations"}
)

# Summary
print("\n" + "=" * 55)
print("INGESTION SUMMARY")
print("=" * 55)
total_vectors = 0
for r in results:
    print(f"  {r['file']:<35} {r['chunks']:>3} chunks  {r['vectors_stored']:>3} vectors")
    total_vectors += r['vectors_stored']
print(f"{'─'*55}")
print(f"  {'TOTAL':<35} {sum(r['chunks'] for r in results):>3} chunks  {total_vectors:>3} vectors")

In [ ]:
# ── Test: Search across all indexed documents ─────────────────────────────────

test_queries = [
    "What are the Q1 revenue figures by region?",
    "Who is responsible for hiring ML engineers?",
    "What is the data retention policy?",
    "Which employees work in engineering?",
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"🔍 {query}")
    print(f"{'='*60}")
    results = pipeline.search(query, top_k=2)
    for r in results:
        print(f"  [{r['distance']:.4f}] {r['metadata']['source']} — '{r['metadata'].get('heading', 'N/A')}'")
        print(f"  {r['metadata']['text'][:150]}...\n")

In [ ]:
# ── Grounded Q&A (Preview of RAG) ─────────────────────────────────────────────
# This is what Session 4 will fully develop!

questions = [
    "What was the total Q1 revenue and which region grew the most?",
    "What action items came out of the product roadmap meeting and who owns them?",
    "What are the API endpoints available in the AI Platform?",
]

for question in questions:
    print(f"\n{'='*65}")
    print(f"❓ {question}")
    print(f"{'='*65}")
    answer = pipeline.ask_document(question, top_k=3)
    print(f"💬 {answer}")

In [ ]:
# ── Filtered search: only search within a specific file type ──────────────────
print("Filtered search — Excel documents only:")
results = pipeline.search(
    "employee salary information",
    top_k=3,
    filter={"file_type": "xlsx"}
)
for r in results:
    print(f"  [{r['distance']:.4f}] {r['metadata']['source']}")
    print(f"  {r['metadata']['text'][:200]}\n")

---

## 🎉 Summary & What's Next

You've built a complete document processing pipeline:

| Stage | Tool / Technique | Key Takeaway |
|---|---|---|
| **Parse** | `markitdown`, `pymupdf4llm` | Every file type → clean Markdown |
| **Extract** | LLM + JSON schema | Unstructured text → typed data |
| **Chunk** | Fixed / sentence / header / page | Chunking strategy affects retrieval quality |
| **Embed** | `text-embedding-3-small` | Semantic vectors for each chunk |
| **Store** | S3 Vectors / LocalVectorStore | Indexed for sub-second ANN search |
| **Search** | Cosine similarity + metadata filter | Retrieve the most relevant chunks |

### 🚀 Coming in Session 4: RAG (Retrieval-Augmented Generation)

You've seen a preview of RAG in `ask_document()`. Session 4 will go much deeper:

- **Query rewriting** — improve retrieval by rephrasing questions
- **Hybrid search** — combine semantic + keyword search
- **Re-ranking** — re-score retrieved chunks before sending to LLM
- **Grounding & citations** — force the LLM to cite its sources
- **Evaluation** — measure RAG quality (precision, recall, faithfulness)
- **Multi-document RAG** — answer across a large corpus

### 📚 Key Resources
- 📖 [markitdown GitHub](https://github.com/microsoft/markitdown)
- 📖 [pymupdf4llm Documentation](https://pymupdf.readthedocs.io/en/latest/pymupdf4llm/)
- 📖 [Amazon S3 Vectors Guide](https://docs.aws.amazon.com/AmazonS3/latest/userguide/s3-vectors.html)
- 📖 [OpenAI Embeddings Guide](https://platform.openai.com/docs/guides/embeddings)